# 🧠 Quantum LSTM — FTD→AD Progression Modeling
### Adapted to OpenNeuro ds005589 EEG Dataset

---

> **⚠️ Dataset Adaptation Note**  
> ds005589 is an **EEG visual object recognition** dataset (healthy subjects, 32 participants, 100 sessions).  
> It does **not** contain clinical FTD/AD labels or fMRI/DWI modalities.  
> **Adaptation strategy:**
> - EEG spectral power features (delta/theta/alpha/beta/gamma bands) → feature vector per session
> - Sessions = longitudinal timepoints (progression trajectory)
> - Behavioral accuracy + session index → proxy CN / FTD / AD stage labels
> - Same Quantum LSTM architecture applied to EEG-derived features

---

## 🗺️ Pipeline Overview
```
Phase 1 → Load .npy EEG data (BIDS derivatives)  →  EDA
Phase 2 → Spectral features + PCA (4 qubits)     →  Sequence windows
Phase 3 → Quantum LSTM training                  →  CN / FTD / AD
Phase 4 → Progression graphs + survival curves   →  Clinical insights
```

---
# 📦 Phase 1 — Setup, Mount & Load

In [ ]:
# ── Install required libraries ────────────────────────────────────────────────
!pip install pennylane pennylane-lightning lifelines scikit-learn torch -q

import os, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from pathlib import Path
from scipy.signal import welch
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
import pennylane as qml
from lifelines import KaplanMeierFitter
warnings.filterwarnings('ignore')
print('✅ All libraries loaded.')

In [ ]:
# ── Mount Drive & set paths ───────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

# ✏️ Change to YOUR Drive folder
DATASET_ROOT  = '/content/drive/MyDrive/DATASET'
PREPROCESSED  = os.path.join(DATASET_ROOT, 'derivatives', 'preprocessed_data')
ANNOTATIONS   = os.path.join(DATASET_ROOT, 'derivatives', 'annotations')

subjects = sorted([d for d in os.listdir(PREPROCESSED) if d.startswith('sub-')])
print(f'📌 Subjects found: {len(subjects)}')
print(f'   First 5: {subjects[:5]}')

In [ ]:
# ── Load subject info (demographics + accuracy) ───────────────────────────────
subject_info = pd.read_csv(os.path.join(ANNOTATIONS, 'Subject_info.csv'))
subject_info.columns = subject_info.columns.str.strip()

# ── Proxy label assignment ────────────────────────────────────────────────────
# High accuracy (≥98.5%) → CN (Cognitively Normal)
# Medium accuracy (97–98.5%) → FTD proxy
# Lower accuracy (<97%) → AD proxy
# NOTE: This is a SIMULATION — real labels require clinical diagnosis.
def assign_stage(rsvp_acc):
    if rsvp_acc >= 98.5: return 'CN'
    elif rsvp_acc >= 97:  return 'FTD'
    else:                 return 'AD'

subject_info['Stage'] = subject_info['RSVP-accuracy (%)'].apply(assign_stage)
print('=== Subject Stage Assignments ===')
display(subject_info[['Participant ID', 'Age', 'Sex', 'RSVP-accuracy (%)', 'Stage']].head(12))
print('\nStage distribution:')
print(subject_info['Stage'].value_counts())

In [ ]:
# ── EDA Visualization 1: Demographics & Stage Distribution ───────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle('Phase 1 — Dataset EDA: Demographics & Stage Labels', fontsize=13, fontweight='bold')

# Age by stage
stage_colors = {'CN': '#2ecc71', 'FTD': '#f39c12', 'AD': '#e74c3c'}
for stage, grp in subject_info.groupby('Stage'):
    axes[0].hist(grp['Age'], bins=8, alpha=0.7, label=stage, color=stage_colors[stage], edgecolor='white')
axes[0].set_title('Age Distribution by Stage')
axes[0].set_xlabel('Age'); axes[0].set_ylabel('Count'); axes[0].legend()

# Stage pie chart
counts = subject_info['Stage'].value_counts()
axes[1].pie(counts, labels=counts.index, colors=[stage_colors[s] for s in counts.index],
            autopct='%1.0f%%', startangle=90)
axes[1].set_title('Stage Distribution (Proxy Labels)')

# RSVP accuracy by stage (boxplot)
order = ['CN', 'FTD', 'AD']
data_by_stage = [subject_info[subject_info['Stage'] == s]['RSVP-accuracy (%)'].values for s in order]
bp = axes[2].boxplot(data_by_stage, labels=order, patch_artist=True)
for patch, stage in zip(bp['boxes'], order):
    patch.set_facecolor(stage_colors[stage])
axes[2].set_title('RSVP Accuracy per Stage')
axes[2].set_xlabel('Stage'); axes[2].set_ylabel('Accuracy (%)')

plt.tight_layout()
plt.savefig('phase1_eda.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Phase 1 EDA complete.')

---
# 🔬 Phase 2 — Feature Extraction, PCA → 4 Qubits, Sequence Building

In [ ]:
# ── EEG Band Power Feature Extractor ─────────────────────────────────────────
SFREQ = 1000  # Hz
BANDS = {'delta': (0.5, 4), 'theta': (4, 8),
         'alpha': (8, 13),  'beta':  (13, 30), 'gamma': (30, 45)}

def band_power(data_1d):
    """Compute relative band power for one channel time-series."""
    freqs, psd = welch(data_1d, fs=SFREQ, nperseg=256)
    total = psd.sum() + 1e-12
    powers = []
    for flo, fhi in BANDS.values():
        mask = (freqs >= flo) & (freqs <= fhi)
        powers.append(psd[mask].sum() / total)
    return np.array(powers)   # shape (5,)

def extract_session_features(subject, session, task='rsvp',
                              runs=['run-01','run-02','run-03','run-04'],
                              n_channels=8):
    """
    Load all runs for one session → mean band power across trials & first N channels.
    Returns feature vector of shape (5 bands × n_channels,) = 40-D.
    """
    all_powers = []
    base = os.path.join(PREPROCESSED, subject, session)
    for run in runs:
        stem = f'{subject}_{session}_task-{task}_{run}_1000Hz.npy'
        path = os.path.join(base, stem)
        if not os.path.exists(path): continue
        data = np.load(path, allow_pickle=True)   # (trials, channels, time)
        # Use first n_channels, mean trial
        mean_trial = data.mean(axis=0)[:n_channels, :]  # (n_ch, T)
        run_powers = np.stack([band_power(mean_trial[c]) for c in range(mean_trial.shape[0])])
        all_powers.append(run_powers.mean(axis=0))  # (5,)
    if not all_powers:
        return None
    return np.stack(all_powers).mean(axis=0)   # (5,) averaged across runs

print('Feature extractor ready.')

In [ ]:
# ── Build longitudinal feature matrix ─────────────────────────────────────────
# Rows = (subject, session) pairs; Columns = 5 band power features
records = []
for sub in subjects:
    sub_path = os.path.join(PREPROCESSED, sub)
    if not os.path.isdir(sub_path): continue
    sessions = sorted(os.listdir(sub_path))
    # Look up stage label
    row = subject_info[subject_info['Participant ID'].str.strip() == sub]
    stage = row['Stage'].values[0] if len(row) else 'CN'
    for ses_idx, ses in enumerate(sessions):
        feat = extract_session_features(sub, ses)
        if feat is not None:
            records.append({'subject': sub, 'session': ses,
                            'ses_idx': ses_idx, 'stage': stage,
                            **{b: v for b, v in zip(BANDS.keys(), feat)}})

df = pd.DataFrame(records)
print(f'✅ Feature matrix built: {df.shape}')
print(f'   Subjects: {df.subject.nunique()}  |  Stage counts: {df.stage.value_counts().to_dict()}')
display(df.head(8))

In [ ]:
# ── PCA → 4 components (4 qubits) ────────────────────────────────────────────
feature_cols = list(BANDS.keys())
X_raw = df[feature_cols].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)

pca = PCA(n_components=4)
X_pca = pca.fit_transform(X_scaled)   # (N, 4) → 4 qubits

print(f'PCA explained variance per component: {pca.explained_variance_ratio_.round(3)}')
print(f'Total variance retained (4 qubits): {pca.explained_variance_ratio_.sum():.1%}')

# Normalize PCA components to [0, π] for quantum angle embedding
X_min, X_max = X_pca.min(0), X_pca.max(0)
X_quantum = np.pi * (X_pca - X_min) / (X_max - X_min + 1e-8)

# ── Visualization 2: PCA Space ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Phase 2 — PCA → 4-Qubit Feature Space', fontsize=13, fontweight='bold')

# Scree plot
axes[0].bar(range(1, 5), pca.explained_variance_ratio_,
            color=['#3498db','#e74c3c','#2ecc71','#f39c12'], edgecolor='white')
axes[0].plot(range(1, 5), np.cumsum(pca.explained_variance_ratio_),
             'ko--', markersize=6, label='Cumulative')
axes[0].set_title('Explained Variance (Scree Plot)')
axes[0].set_xlabel('Principal Component (Qubit)'); axes[0].set_ylabel('Variance Ratio')
axes[0].legend(); axes[0].set_xticks(range(1, 5))

# PC1 vs PC2 colored by stage
for stage, c in stage_colors.items():
    mask = df['stage'] == stage
    axes[1].scatter(X_pca[mask, 0], X_pca[mask, 1], c=c, label=stage, alpha=0.7, edgecolors='white', s=60)
axes[1].set_title('PC1 vs PC2 — Stage Separation')
axes[1].set_xlabel('PC1 (Qubit 0)'); axes[1].set_ylabel('PC2 (Qubit 1)')
axes[1].legend()

plt.tight_layout()
plt.savefig('phase2_pca.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ PCA complete. X_quantum shape:', X_quantum.shape)

In [ ]:
# ── Build Sliding Window Sequences per Subject ────────────────────────────────
WINDOW = 3   # timesteps (sessions)

le = LabelEncoder()
df['label'] = le.fit_transform(df['stage'])   # CN=0, FTD=1, AD=2
print(f'Label mapping: {dict(zip(le.classes_, le.transform(le.classes_)))}')

seq_X, seq_y = [], []
for sub in df['subject'].unique():
    sub_df = df[df['subject'] == sub].sort_values('ses_idx')
    vals  = X_quantum[sub_df.index]
    labs  = sub_df['label'].values
    for i in range(len(vals) - WINDOW + 1):
        seq_X.append(vals[i:i + WINDOW])   # (window, 4)
        seq_y.append(labs[i + WINDOW - 1]) # label at last step

seq_X = np.array(seq_X, dtype=np.float32)   # (N_seq, WINDOW, 4)
seq_y = np.array(seq_y, dtype=np.int64)
print(f'Sequences: X={seq_X.shape}, y={seq_y.shape}')
print(f'Label counts: {pd.Series(seq_y).map({i:s for i,s in enumerate(le.classes_)}).value_counts().to_dict()}')

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    seq_X, seq_y, test_size=0.25, random_state=42, stratify=seq_y
)
print(f'Train: {X_train.shape}  |  Test: {X_test.shape}')

---
# ⚛️ Phase 3 — Quantum LSTM Model

In [ ]:
# ── Quantum Circuit: 4-qubit gate ─────────────────────────────────────────────
N_QUBITS = 4
N_LAYERS = 2

dev = qml.device('default.qubit', wires=N_QUBITS)

@qml.qnode(dev, interface='torch')
def quantum_gate(inputs, weights):
    """AngleEmbedding + StronglyEntanglingLayers → PauliZ expectations"""
    qml.AngleEmbedding(inputs, wires=range(N_QUBITS), rotation='X')
    qml.StronglyEntanglingLayers(weights, wires=range(N_QUBITS))
    return [qml.expval(qml.PauliZ(i)) for i in range(N_QUBITS)]

weight_shape = qml.StronglyEntanglingLayers.shape(n_layers=N_LAYERS, n_wires=N_QUBITS)
print(f'Quantum gate weight shape: {weight_shape}')
print(qml.draw(quantum_gate)(np.zeros(N_QUBITS), np.zeros(weight_shape)))

In [ ]:
# ── Quantum LSTM Cell ─────────────────────────────────────────────────────────
class QLSTMCell(nn.Module):
    """One QLSTM step: 4 quantum gates (forget / input / candidate / output)"""
    def __init__(self, input_size=4, hidden_size=4):
        super().__init__()
        self.hidden_size = hidden_size
        self.proj   = nn.Linear(input_size + hidden_size, N_QUBITS)
        ws = weight_shape
        # Four quantum gates
        self.w_f = nn.Parameter(torch.randn(ws) * 0.1)
        self.w_i = nn.Parameter(torch.randn(ws) * 0.1)
        self.w_c = nn.Parameter(torch.randn(ws) * 0.1)
        self.w_o = nn.Parameter(torch.randn(ws) * 0.1)

    def _qgate(self, x, weights):
        return torch.stack([quantum_gate(x[b], weights) for b in range(x.shape[0])])

    def forward(self, x, h, c):
        # Project concatenated input+hidden to 4-qubit space
        combined = self.proj(torch.cat([x, h], dim=-1))  # (B, 4)
        combined = torch.sigmoid(combined) * np.pi        # scale to [0, π]

        f = torch.sigmoid(self._qgate(combined, self.w_f))  # forget
        i = torch.sigmoid(self._qgate(combined, self.w_i))  # input
        g = torch.tanh  (self._qgate(combined, self.w_c))   # candidate
        o = torch.sigmoid(self._qgate(combined, self.w_o))  # output

        c_new = f * c + i * g
        h_new = o * torch.tanh(c_new)
        return h_new, c_new


class QLSTM(nn.Module):
    """Full Quantum LSTM: unroll over timesteps → classifier head"""
    def __init__(self, input_size=4, hidden_size=4, n_classes=3):
        super().__init__()
        self.cell       = QLSTMCell(input_size, hidden_size)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, 16),
            nn.ReLU(),
            nn.Linear(16, n_classes)
        )

    def forward(self, x):
        B, T, _ = x.shape
        h = torch.zeros(B, self.cell.hidden_size)
        c = torch.zeros(B, self.cell.hidden_size)
        for t in range(T):
            h, c = self.cell(x[:, t, :], h, c)
        return self.classifier(h)


model = QLSTM(input_size=N_QUBITS, hidden_size=N_QUBITS, n_classes=3)
total_params = sum(p.numel() for p in model.parameters())
print(f'✅ QLSTM initialized  |  Total params: {total_params}')
print(model)

In [ ]:
# ── Training Loop ─────────────────────────────────────────────────────────────
EPOCHS    = 40
LR        = 0.01
BATCH     = 8
N_CLASSES = 3

Xt = torch.tensor(X_train)
yt = torch.tensor(y_train)
Xv = torch.tensor(X_test)
yv = torch.tensor(y_test)

optimizer   = torch.optim.Adam(model.parameters(), lr=LR)
criterion   = nn.CrossEntropyLoss()

train_losses, val_losses = [], []
train_accs,   val_accs   = [], []

for epoch in range(EPOCHS):
    model.train()
    perm   = torch.randperm(Xt.shape[0])
    epoch_loss = 0
    for i in range(0, Xt.shape[0], BATCH):
        idx     = perm[i:i+BATCH]
        xb, yb  = Xt[idx], yt[idx]
        out     = model(xb)
        loss    = criterion(out, yb)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()

    # Metrics
    model.eval()
    with torch.no_grad():
        tr_pred = model(Xt).argmax(1); tr_acc = (tr_pred == yt).float().mean().item()
        vl_pred = model(Xv).argmax(1); vl_acc = (vl_pred == yv).float().mean().item()
        vl_loss = criterion(model(Xv), yv).item()

    train_losses.append(epoch_loss / max(1, Xt.shape[0]//BATCH))
    val_losses.append(vl_loss)
    train_accs.append(tr_acc)
    val_accs.append(vl_acc)

    if (epoch+1) % 10 == 0:
        print(f'Epoch {epoch+1:3d}/{EPOCHS} | '
              f'Train Loss: {train_losses[-1]:.3f} Acc: {tr_acc:.2%} | '
              f'Val Loss: {vl_loss:.3f} Acc: {vl_acc:.2%}')

print('\n✅ Training complete!')

In [ ]:
# ── Visualization 3: Training Dashboard ──────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Phase 3 — QLSTM Training Dashboard', fontsize=13, fontweight='bold')

axes[0].plot(train_losses, label='Train Loss', color='#3498db')
axes[0].plot(val_losses,   label='Val Loss',   color='#e74c3c', linestyle='--')
axes[0].set_title('Loss Curves'); axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot([a*100 for a in train_accs], label='Train Acc', color='#2ecc71')
axes[1].plot([a*100 for a in val_accs],   label='Val Acc',   color='#f39c12', linestyle='--')
axes[1].axhline(90, color='gray', linestyle=':', linewidth=1, label='90% target')
axes[1].set_title('Accuracy Curves'); axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy (%)')
axes[1].set_ylim(0, 105); axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('phase3_training.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Evaluation: Confusion Matrix + Classification Report ─────────────────────
model.eval()
with torch.no_grad():
    y_pred = model(Xv).argmax(1).numpy()

print('=== Classification Report ===')
print(classification_report(y_test, y_pred, target_names=le.classes_))

fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(
    confusion_matrix(y_test, y_pred),
    display_labels=le.classes_
).plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title('QLSTM Confusion Matrix (CN / FTD / AD)')
plt.tight_layout()
plt.savefig('phase3_confusion.png', dpi=150, bbox_inches='tight')
plt.show()

---
# 📈 Phase 4 — Progression Graphs & Survival Analysis

In [ ]:
# ── Compute per-subject progression probabilities over sessions ───────────────
model.eval()
softmax = nn.Softmax(dim=1)

prog_records = []
for sub in df['subject'].unique():
    sub_df = df[df['subject'] == sub].sort_values('ses_idx')
    vals   = X_quantum[sub_df.index]
    stage  = sub_df['stage'].iloc[0]

    for i in range(len(vals) - WINDOW + 1):
        seq   = torch.tensor(vals[i:i+WINDOW][None], dtype=torch.float32)
        with torch.no_grad():
            probs = softmax(model(seq)).squeeze().numpy()
        prog_records.append({
            'subject':    sub,
            'stage':      stage,
            'timepoint':  i + WINDOW,
            'P(CN)':      probs[le.transform(['CN'])[0]],
            'P(FTD)':     probs[le.transform(['FTD'])[0]],
            'P(AD)':      probs[le.transform(['AD'])[0]]
        })

prog_df = pd.DataFrame(prog_records)
print(f'Progression records: {len(prog_df)}')
display(prog_df.head(8))

In [ ]:
# ── Visualization 4A: Patient Cohort Progression Heatmap ─────────────────────
pivot = prog_df.groupby(['subject','timepoint'])['P(AD)'].mean().unstack(fill_value=np.nan)

fig, ax = plt.subplots(figsize=(14, 8))
sns.heatmap(pivot, ax=ax, cmap='RdYlGn_r', vmin=0, vmax=1,
            linewidths=0.3, linecolor='white',
            cbar_kws={'label': 'P(AD) — Alzheimer Stage Probability'})
ax.set_title('Phase 4 — Patient Cohort Progression Heatmap\nP(AD) per Subject × Timepoint',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Timepoint (Session Window)')
ax.set_ylabel('Subject')
plt.tight_layout()
plt.savefig('phase4_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Visualization 4B: Mean Progression Trajectories by Stage ─────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)
fig.suptitle('Phase 4 — Mean Stage Probability Trajectories per Cohort',
             fontsize=13, fontweight='bold')

for ax, cohort in zip(axes, ['CN','FTD','AD']):
    sub_prog = prog_df[prog_df['stage'] == cohort]
    for col, c in [('P(CN)','#2ecc71'), ('P(FTD)','#f39c12'), ('P(AD)','#e74c3c')]:
        mean_traj = sub_prog.groupby('timepoint')[col].mean()
        std_traj  = sub_prog.groupby('timepoint')[col].std().fillna(0)
        ax.plot(mean_traj.index, mean_traj.values, label=col, color=c, linewidth=2, marker='o', markersize=4)
        ax.fill_between(mean_traj.index,
                        mean_traj - std_traj, mean_traj + std_traj,
                        alpha=0.15, color=c)
    ax.set_title(f'{cohort} Cohort (n={sub_prog.subject.nunique()})')
    ax.set_xlabel('Timepoint'); ax.set_ylabel('Probability')
    ax.set_ylim(0, 1); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('phase4_trajectories.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Visualization 4C: Kaplan-Meier Style Survival Curves ─────────────────────
# "Event" = P(AD) crossing 0.5 threshold for the first time
AD_THRESHOLD = 0.5
survival_data = []
for sub in prog_df['subject'].unique():
    sub_data = prog_df[prog_df['subject'] == sub].sort_values('timepoint')
    stage    = sub_data['stage'].iloc[0]
    crossed  = sub_data[sub_data['P(AD)'] >= AD_THRESHOLD]
    if len(crossed):
        survival_data.append({'subject': sub, 'stage': stage,
                              'time_to_event': crossed['timepoint'].min(), 'event': 1})
    else:
        survival_data.append({'subject': sub, 'stage': stage,
                              'time_to_event': sub_data['timepoint'].max(), 'event': 0})

surv_df = pd.DataFrame(survival_data)

fig, ax = plt.subplots(figsize=(10, 6))
for cohort, c in [('CN','#2ecc71'), ('FTD','#f39c12'), ('AD','#e74c3c')]:
    grp = surv_df[surv_df['stage'] == cohort]
    if len(grp) < 2: continue
    kmf = KaplanMeierFitter()
    kmf.fit(grp['time_to_event'], event_observed=grp['event'], label=f'{cohort} (n={len(grp)})')
    kmf.plot_survival_function(ax=ax, color=c, linewidth=2.5, ci_show=True)

ax.set_title('Phase 4 — Kaplan-Meier: Time to P(AD) > 0.5 Threshold',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Session Timepoint'); ax.set_ylabel('Proportion NOT reaching AD stage')
ax.axhline(0.5, color='gray', linestyle='--', linewidth=1, label='50% threshold')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('phase4_survival.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Visualization 4D: Quantum State Evolution (expectation values during training) ──
# Re-run forward pass across test set, collect qubit expectation values
model.eval()
qubit_vals_per_class = {cls: [] for cls in le.classes_}

with torch.no_grad():
    for idx in range(min(len(Xv), 40)):
        x   = Xv[idx:idx+1]   # (1, T, 4)
        # Get quantum cell output after 1 step
        h = torch.zeros(1, N_QUBITS)
        c = torch.zeros(1, N_QUBITS)
        for t in range(WINDOW):
            combined = model.cell.proj(torch.cat([x[:,t,:], h], dim=-1))
            combined = torch.sigmoid(combined) * np.pi
            # Record forget gate quantum output
            qout = quantum_gate(combined[0], model.cell.w_f).detach().numpy()
            h, c = model.cell(x[:,t,:], h, c)
        stage_pred = le.classes_[y_test[idx]]
        qubit_vals_per_class[stage_pred].append(qout)

fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)
fig.suptitle('Phase 4 — Quantum Gate Expectation Values ⟨Z⟩ per Stage',
             fontsize=13, fontweight='bold')

for ax, (stage, c) in zip(axes, stage_colors.items()):
    vals = qubit_vals_per_class.get(stage, [])
    if not vals:
        ax.set_title(f'{stage} (no samples)'); continue
    arr = np.array(vals)   # (n_samples, 4)
    ax.bar(range(N_QUBITS), arr.mean(0), yerr=arr.std(0),
           color=c, alpha=0.8, edgecolor='white', capsize=5)
    ax.set_title(f'{stage} Stage')
    ax.set_xlabel('Qubit index'); ax.set_ylabel('⟨PauliZ⟩')
    ax.set_xticks(range(N_QUBITS))
    ax.set_xticklabels([f'Q{i}' for i in range(N_QUBITS)])
    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_ylim(-1.2, 1.2); ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('phase4_quantum_states.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Final Summary Dashboard ───────────────────────────────────────────────────
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import label_binarize

model.eval()
with torch.no_grad():
    logits    = model(Xv)
    probs_all = softmax(logits).numpy()
    y_pred_f  = logits.argmax(1).numpy()

y_bin = label_binarize(y_test, classes=range(3))
try:
    auc = roc_auc_score(y_bin, probs_all, multi_class='ovr', average='macro')
except:
    auc = float('nan')

final_acc = (y_pred_f == y_test).mean()

print('=' * 50)
print('        QLSTM FINAL RESULTS SUMMARY')
print('=' * 50)
print(f'  Subjects       : {df.subject.nunique()}')
print(f'  Sessions total : {len(df)}')
print(f'  Sequences      : {len(seq_X)} (window={WINDOW})')
print(f'  Epochs trained : {EPOCHS}')
print(f'  Val Accuracy   : {final_acc:.1%}')
print(f'  Val AUC (OvR)  : {auc:.3f}')
print(f'  Best Val Acc   : {max(val_accs):.1%} @ epoch {val_accs.index(max(val_accs))+1}')
print('=' * 50)
print('\nVisualisations saved:')
for f in ['phase1_eda.png','phase2_pca.png','phase3_training.png',
          'phase3_confusion.png','phase4_heatmap.png',
          'phase4_trajectories.png','phase4_survival.png','phase4_quantum_states.png']:
    print(f'  ✅ {f}')

---
## 📋 Architecture Summary

| Component | Detail |
|-----------|--------|
| **Input** | EEG band power features (5 bands × 8 channels → PCA → 4D) |
| **Sequence window** | 3 sessions (longitudinal timepoints) |
| **Quantum circuit** | 4 qubits · AngleEmbedding (RX) · StronglyEntanglingLayers (2 layers) |
| **LSTM gates** | 4 quantum gates (forget / input / candidate / output) |
| **Classifier** | FC(4→16→3) + Softmax → CN / FTD / AD |
| **Loss** | CrossEntropyLoss |
| **Optimizer** | Adam (lr=0.01) |
| **Labels** | Proxy: accuracy ≥98.5%→CN, 97–98.5%→FTD, <97%→AD |

> **⚠️ Clinical Disclaimer:** Labels and progression curves are **proxies derived from EEG task accuracy**, not clinical diagnoses. Real FTD/AD modeling requires longitudinal clinical cohorts with confirmed diagnoses.